In [1]:
import torch
from sklearn.model_selection import train_test_split
import yaml
import argparse
import shutil
from collections import namedtuple
import os
import datetime


# Custom Libraries
from utils.data_generator import DataGenerator
from utils.agent import TrainModel
from utils.helper import to_python_native, gen_experiment_name, set_seed, save_model_state
from utils.pennylane.model import Qkernel

In [2]:
# === Backend Configuration ===
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.xpu.is_available():
    device = torch.device("xpu")
else:
    device = torch.device("cpu")

set_seed(42)
device

device(type='mps')

In [3]:
data_generator = DataGenerator(
        dataset_name= 'checkerboard',
        file_path='data/checkerboard_dataset.npy',
    )

training_data, training_labels, testing_data, testing_labels = data_generator.generate_dataset()
training_data = torch.tensor(training_data.to_numpy(), dtype=torch.float32, requires_grad=True)
testing_data = torch.tensor(testing_data.to_numpy(), dtype=torch.float32, requires_grad=True)
training_labels = torch.tensor(training_labels.to_numpy(), dtype=torch.int)
testing_labels = torch.tensor(testing_labels.to_numpy(), dtype=torch.int)

In [4]:
print(training_data.size())
print(testing_data.size())

torch.Size([30, 2])
torch.Size([30, 2])


In [1]:
kernel = Qkernel(
        device='lightning.qubit',
        n_qubits=4,
        trainable=True,
        input_scaling=True,
        data_reuploading=True,
        ansatz='embedding_paper',
        ansatz_layers=5,
        noisy=False,
        noise_prob = 0.2,
        diff_method= 'adjoint'
    )

NameError: name 'Qkernel' is not defined

In [6]:
agent = TrainModel(
        kernel=kernel,
        training_data=training_data,
        training_labels=training_labels,
        testing_data=testing_data,
        testing_labels=testing_labels,
        optimizer='gd',
        lr= 0.1,
        mclr=0.01,
        cclr=0.01,
        epochs=100,
        train_method='ccka',
        target_accuracy=0.95,
        get_alignment_every=1000,
        validate_every_epoch=1,
        base_path='../',
        lambda_kao=0.001,
        lambda_co=0.001,
        clusters=2,
        use_kmeans=True
    )

Epochs:  10


In [7]:
before_metrics = agent.evaluate(testing_data, testing_labels, 'before')
before_metrics

{'alignment': tensor(0.1206, grad_fn=<SqueezeBackward0>),
 'executions': None,
 'training_accuracy': 0.8333333333333334,
 'testing_accuracy': 0.8,
 'f1_score': 0.7945701357466064,
 'alignment_arr': [[], []],
 'loss_arr': [],
 'validation_accuracy_arr': []}

In [8]:
 agent.fit_multiclass(training_data, training_labels)

Started Training


(Qkernel(),
 [Parameter containing:
  tensor([[0.9228, 0.6613, 0.9362, 0.8604],
          [1.1118, 0.9880, 0.9596, 0.7928],
          [0.8645, 1.0778, 0.9263, 1.1075],
          [0.7734, 1.2904, 1.0898, 0.7687],
          [1.2473, 0.6987, 1.0626, 0.9762]], requires_grad=True),
  Parameter containing:
  tensor([[ 2.7326,  2.5408, -0.6237,  2.6036],
          [-0.5469,  0.7233, -1.5951,  2.0248],
          [ 2.6926, -2.5370,  2.7102,  0.6217],
          [ 2.4310,  0.3577,  1.5817, -0.4397],
          [ 2.3268,  0.4461, -1.4091,  0.7694]], requires_grad=True),
  Parameter containing:
  tensor([[-1.3945, -0.2680, -1.2216,  2.0340],
          [-2.2147, -1.3956, -0.8176, -1.9035],
          [ 0.3506, -2.8376,  2.8826, -2.6886],
          [ 2.2300,  0.4972, -0.7829,  1.9215],
          [ 0.4704,  2.4388,  0.3300, -0.9519]], requires_grad=True)],
 ParameterList(
     (0): Parameter containing: [torch.float32 of size 2]
     (1): Parameter containing: [torch.float32 of size 2]
 ),
 ParameterLis

In [9]:
after_metrics = agent.evaluate(testing_data, testing_labels, 'after')
after_metrics

{'alignment': tensor(0.2225, grad_fn=<SqueezeBackward0>),
 'executions': 800,
 'training_accuracy': 1.0,
 'testing_accuracy': 0.9666666666666667,
 'f1_score': 0.9667037449017427,
 'alignment_arr': [[], []],
 'loss_arr': [],
 'validation_accuracy_arr': []}